In [1]:
import requests
import pdfplumber
import pandas as pd
import re
from datetime import datetime, timedelta

# -----------------------------
# CONFIG
# -----------------------------
START_DATE = "2026-04-07" #choose date
END_DATE = "2026-04-12" #choose date
OUTPUT_CSV = "nba_injury_reports_test.csv" #choose output file name
BASE_URL = "https://ak-static.cms.nba.com/referee/injury/Injury-Report_{filename}.pdf"

# Regex patterns
date_pattern = r"\d{2}/\d{2}/\d{4}"
time_pattern_new = r"\d{2}:\d{2}\s*\(ET\)"
time_pattern_old = r"\d{2}:\d{2}[AP]M"
matchup_pattern = r"[A-Z]{2,3}\s*@\s*[A-Z]{2,3}"
player_pattern = r"[A-Za-z\-']+(?:\s*(?:Sr\.|Jr\.|II|III|IV))?,\s*[A-Za-z\.\-']+"

# -----------------------------
# UTILITY FUNCTIONS
# -----------------------------
def download_pdf(url, path):
    try:
        r = requests.get(url)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
        return True
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return False

def normalize_reason(reason):
    """
    Normalize 'Injury/Illness' reason strings to format:
    Injury/Illness - BodyPart; Type
    """
    reason = reason.strip()
    if reason.startswith("Injury/Illness") and "-" not in reason:
        # Attempt to insert dash after 'Injury/Illness'
        parts = reason.replace("Injury/Illness", "").strip().split(";", 1)
        body_part = parts[0] if parts else "N/A"
        injury_type = parts[1].strip() if len(parts) > 1 else "N/A"
        reason = f"Injury/Illness - {body_part}; {injury_type}"
    return reason

def parse_pdf(pdf_path, pdf_date):
    rows = []
    last_row = None
    current_game = {"Game Date": None, "Game Time": None, "Matchup": None, "Team": None}
    pending_reason = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            lines = text.split("\n")

            for line in lines:
                line = re.sub(r"\s+", " ", line).strip()
                if any(x in line for x in ["Game Date", "Page", "Injury Report"]):
                    continue

                date_match = re.search(date_pattern, line)
                matchup_match = re.search(matchup_pattern, line)

                # New game row with date and matchup
                if date_match and matchup_match:
                    game_date = date_match.group()
                    # Stop if game date exceeds PDF release date
                    if datetime.strptime(game_date, "%m/%d/%Y") > pdf_date:
                        return rows

                    time_match = re.search(time_pattern_new, line) or re.search(time_pattern_old, line)
                    game_time = time_match.group().replace(" ", "") if time_match else ""
                    matchup = matchup_match.group()
                    after_matchup = line.split(matchup)[1].strip()
                    player_match = re.search(player_pattern, after_matchup)
                    if not player_match:
                        continue
                    player_start = player_match.start()
                    team = after_matchup[:player_start].strip()
                    player = player_match.group().strip()
                    rest = after_matchup[player_match.end():].strip()
                    parts = rest.split(" ", 1)
                    status = parts[0]
                    reason = parts[1] if len(parts) > 1 else ""

                    if pending_reason:
                        reason = pending_reason + " " + reason
                        pending_reason = ""

                    current_game = {
                        "Game Date": game_date,
                        "Game Time": game_time,
                        "Matchup": matchup.replace(" ", ""),
                        "Team": team
                    }

                    row = {
                        **current_game,
                        "Player Name": player,
                        "Current Status": status,
                        "Reason": normalize_reason(reason)
                    }
                    rows.append(row)
                    last_row = row
                    continue

                # New team row with same matchup
                elif matchup_match:
                    matchup = matchup_match.group()
                    after_matchup = line.split(matchup)[1].strip()
                    player_match = re.search(player_pattern, after_matchup)
                    if not player_match:
                        continue
                    player_start = player_match.start()
                    team = after_matchup[:player_start].strip()
                    current_game["Matchup"] = matchup.replace(" ", "")
                    current_game["Team"] = team

                    player = player_match.group().strip()
                    rest = after_matchup[player_match.end():].strip()
                    parts = rest.split(" ", 1)
                    status = parts[0]
                    reason = parts[1] if len(parts) > 1 else ""

                    if pending_reason:
                        reason = pending_reason + " " + reason
                        pending_reason = ""

                    row = {
                        **current_game,
                        "Player Name": player,
                        "Current Status": status,
                        "Reason": normalize_reason(reason)
                    }
                    rows.append(row)
                    last_row = row
                    continue

                # Continuation lines for Injury/Illness
                else:
                    player_match = re.search(player_pattern, line)
                    if line.startswith("Injury/Illness") and not player_match:
                        pending_reason += " " + line if pending_reason else line
                        continue
                    elif player_match:
                        player_start = player_match.start()
                        before_player = line[:player_start].strip()
                        if before_player and len(before_player.split()) <= 4:
                            current_game["Team"] = before_player
                        player = player_match.group().strip()
                        rest = line[player_match.end():].strip()
                        parts = rest.split(" ", 1)
                        status = parts[0]
                        reason = parts[1] if len(parts) > 1 else ""

                        if pending_reason:
                            reason = pending_reason + " " + reason
                            pending_reason = ""

                        row = {
                            **current_game,
                            "Player Name": player,
                            "Current Status": status,
                            "Reason": normalize_reason(reason)
                        }
                        rows.append(row)
                        last_row = row
                    else:
                        if last_row:
                            last_row["Reason"] += " " + line

    return rows

def clean_player_name(name):
    name = re.sub(r"\b(Sr|Jr|II|III|IV)\b\.?", "", name, flags=re.IGNORECASE)
    name = name.replace(".", "")
    try:
        last, first = name.split(",")
        return f"{first.strip()} {last.strip()}"
    except:
        return name.strip()

# -----------------------------
# MAIN LOOP
# -----------------------------
all_rows = []
start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
end_dt = datetime.strptime(END_DATE, "%Y-%m-%d")
delta = timedelta(days=1)

current_dt = start_dt
while current_dt <= end_dt:
    date_str = current_dt.strftime("%Y-%m-%d")
    filename = f"{date_str}_11PM" if current_dt < datetime(2025, 12, 22) else f"{date_str}_11_00PM"
    pdf_url = BASE_URL.format(filename=filename)
    pdf_path = f"tmp_{date_str}.pdf"

    print(f"Processing {date_str} ...")
    if download_pdf(pdf_url, pdf_path):
        pdf_date = datetime.strptime(current_dt.strftime("%m/%d/%Y"), "%m/%d/%Y")
        rows = parse_pdf(pdf_path, pdf_date)
        all_rows.extend(rows)
    else:
        print(f"Skipping {date_str} (PDF not found)")

    current_dt += delta

# -----------------------------
# CREATE DATAFRAME
# -----------------------------
df = pd.DataFrame(all_rows)
df = df.dropna(subset=["Player Name"]).reset_index(drop=True)
df["Player Name"] = df["Player Name"].apply(clean_player_name)

# -----------------------------
# SAVE CSV
# -----------------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"All done! Saved {len(df)} rows to {OUTPUT_CSV}")

Processing 2026-04-07 ...
Processing 2026-04-08 ...
Processing 2026-04-09 ...
Processing 2026-04-10 ...
Processing 2026-04-11 ...
Processing 2026-04-12 ...
All done! Saved 710 rows to nba_injury_reports_test.csv
